# 🛠️ Notebook 2: Facebook (core social graph) — Implementation


## 🛠️ Setup

```bash
cd 07-object-oriented-design/facebook
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> This notebook assumes you've read **Notebook 1** — we're now putting the "best" choices from that notebook into one working implementation.


## Step 1 — Enums: `ReactionType` and `Privacy`

Enums give us a **closed set** of values with readable names. If someone tries `post.privacy = 'pubic'` (typo),
Python will happily accept the string — with an enum it would fail fast.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from enum import Enum
from itertools import count

class ReactionType(Enum):
    LIKE  = '👍'
    LOVE  = '❤️'
    HAHA  = '😂'
    WOW   = '😮'
    SAD   = '😢'
    ANGRY = '😡'

class Privacy(Enum):
    PUBLIC  = 'public'    # anyone can see
    FRIENDS = 'friends'   # only friends of author
    ONLY_ME = 'only_me'   # only author

print(list(ReactionType))
print(list(Privacy))


## Step 2 — `User` with symmetric friendship **and** blocking

Real social networks have three states, not two:

- **friends**: mutual, see each other's stuff (by default)
- **neutral**: strangers, can see only public posts
- **blocked**: one-sided but enforced *both ways* (neither can see the other)

Blocking takes priority over friendship — so if Alice blocks Bob, we also drop the friendship edge.


In [ ]:
_uid = count(1)

@dataclass
class User:
    name: str
    id: int = field(default_factory=lambda: next(_uid))
    friends: set['User'] = field(default_factory=set)
    blocked: set['User'] = field(default_factory=set)   # users this user has blocked

    # WHY define __hash__/__eq__ by hand?
    # @dataclass generates __eq__ (field-by-field) and then sets __hash__ = None,
    # which makes the class UNHASHABLE. That is deliberate: a hash must not change
    # while the object sits in a set, and dataclass fields are mutable.
    # Our fix is the standard entity rule: identity is the surrogate id, not the
    # field values. Alice renamed is still Alice, so both __eq__ and __hash__ read
    # only `id`. dataclass leaves hand-written __eq__/__hash__ alone.
    def __hash__(self): return self.id
    def __eq__(self, o): return isinstance(o, User) and self.id == o.id
    def __repr__(self): return f'User({self.name})'

    def add_friend(self, other: 'User') -> None:
        if other is self or other in self.blocked or self in other.blocked:
            return
        self.friends.add(other)
        other.friends.add(self)

    def unfriend(self, other: 'User') -> None:
        self.friends.discard(other)
        other.friends.discard(self)

    def block(self, other: 'User') -> None:
        if other is self: return
        self.blocked.add(other)
        self.unfriend(other)            # blocking implies unfriending

    def unblock(self, other: 'User') -> None:
        self.blocked.discard(other)

    def mutual_friends(self, other: 'User') -> set['User']:
        return self.friends & other.friends   # set intersection

# quick self-check
alice = User('Alice'); bob = User('Bob'); carol = User('Carol')
alice.add_friend(bob); bob.add_friend(carol)
assert alice.mutual_friends(carol) == {bob}
print('Alice and Carol have mutual friends:', alice.mutual_friends(carol))

## Step 3 — `Comment` and `Post` with reactions

We use `dataclass` to skip writing `__init__` / `__repr__` by hand.
`field(default_factory=...)` is the correct pattern for mutable defaults (don't use `= []` or `= {}`).


In [ ]:
_pid = count(1); _cid = count(1)

@dataclass
class Comment:
    author: User
    text: str
    ts: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    id: int = field(default_factory=lambda: next(_cid))

@dataclass
class Post:
    author: User
    content: str
    ts: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    privacy: Privacy = Privacy.FRIENDS
    id: int = field(default_factory=lambda: next(_pid))
    comments: list[Comment] = field(default_factory=list)
    reactions: dict[int, ReactionType] = field(default_factory=dict)  # user.id -> reaction

    def react(self, user: User, r: ReactionType) -> None:
        self.reactions[user.id] = r            # upsert (one reaction per user)

    def remove_reaction(self, user: User) -> None:
        self.reactions.pop(user.id, None)

    def comment(self, user: User, text: str) -> Comment:
        c = Comment(user, text)
        self.comments.append(c)
        return c

    def reaction_summary(self) -> dict[ReactionType, int]:
        out: dict[ReactionType, int] = {}
        for r in self.reactions.values():
            out[r] = out.get(r, 0) + 1
        return out

p = Post(alice, 'hello world', privacy=Privacy.PUBLIC)
p.react(bob, ReactionType.LIKE)
p.react(bob, ReactionType.LOVE)   # bob changed his mind — replaces
p.react(carol, ReactionType.LIKE)
print('reactions on post:', p.reaction_summary())
assert p.reaction_summary() == {ReactionType.LOVE: 1, ReactionType.LIKE: 1}


## Step 4 — Visibility: who can see which post?

A single `can_see` function keeps the rules in one place. Checks in this order:

1. If either user blocked the other → **no**.
2. If the post is `ONLY_ME` → only the author sees it.
3. If the post is `FRIENDS` → author or a friend of the author.
4. `PUBLIC` → anyone not blocked.


In [ ]:
def can_see(viewer: User, post: Post) -> bool:
    author = post.author
    if viewer in author.blocked or author in viewer.blocked:
        return False                       # blocking beats everything
    if post.privacy is Privacy.ONLY_ME:
        return viewer == author
    if post.privacy is Privacy.FRIENDS:
        return viewer == author or viewer in author.friends
    return True                            # PUBLIC

secret = Post(alice, 'my diary', privacy=Privacy.ONLY_ME)
friends_only = Post(alice, 'bday party!', privacy=Privacy.FRIENDS)
public = Post(alice, 'go sports', privacy=Privacy.PUBLIC)

print('Bob (friend) sees diary?     ', can_see(bob, secret))          # False
print('Bob (friend) sees bday?      ', can_see(bob, friends_only))    # True
print('Carol (stranger) sees bday?  ', can_see(carol, friends_only))  # False (not Alice's friend)
print('Carol (stranger) sees sports?', can_see(carol, public))        # True


## Step 5 — `NewsFeed` (fanout-on-read)

**Fanout-on-read** means we compute the feed when the user asks for it, by scanning all posts and
filtering by (a) the author is the viewer or one of their friends, (b) visibility rules.
Simple and always consistent; the cost is paid at read time.


In [ ]:
class NewsFeed:
    def __init__(self, all_posts: list[Post]):
        self._all = all_posts

    def for_user(self, user: User, limit: int = 10) -> list[Post]:
        visible_authors = {user} | user.friends
        return sorted(
            (p for p in self._all if p.author in visible_authors and can_see(user, p)),
            key=lambda p: p.ts,
            reverse=True,
        )[:limit]


## Step 6 — End-to-end demo with assertions

We build a tiny graph, post a few things, react, and check the feed is right.


In [ ]:
# Fresh world so this cell is idempotent
alice = User('Alice'); bob = User('Bob'); carol = User('Carol'); dave = User('Dave')
alice.add_friend(bob)
alice.add_friend(carol)
# Dave is a stranger to everyone

posts: list[Post] = []
now = datetime.now(timezone.utc)

def make(author, content, minutes_ago=0, privacy=Privacy.FRIENDS):
    p = Post(author, content, ts=now - timedelta(minutes=minutes_ago), privacy=privacy)
    posts.append(p)
    return p

p1 = make(alice, 'hello world',               minutes_ago=5)
p2 = make(bob,   'hi alice!',                 minutes_ago=3)
p3 = make(carol, 'new job!',                  minutes_ago=1)
p4 = make(dave,  'nobody knows me',           minutes_ago=2, privacy=Privacy.PUBLIC)
p5 = make(alice, 'my secret diary',           minutes_ago=0, privacy=Privacy.ONLY_ME)

p1.react(bob, ReactionType.LOVE); p1.comment(bob, '❤️')
p2.react(alice, ReactionType.LIKE)

feed = NewsFeed(posts)

alice_feed = feed.for_user(alice)
print("Alice's feed:")
for p in alice_feed:
    print(' ', p.ts.strftime('%H:%M:%S'), p.author, '->', p.content, p.reaction_summary() or '')

# Assertions ---------------------------------------------------
# Alice sees her own secret; friends don't.
assert p5 in alice_feed
assert p5 not in feed.for_user(bob)
# Dave's public post does NOT show in Alice's feed because Dave isn't her friend.
# (Public posts only show up via search / explore — not the friends feed.)
assert p4 not in alice_feed
# Carol sees Alice's friends-only post because they're friends.
assert p1 in feed.for_user(carol)
# Feed is sorted newest-first.
assert alice_feed == sorted(alice_feed, key=lambda p: p.ts, reverse=True)
print('\nall assertions passed')


## Step 7 — Blocking cuts visibility both ways

Blocking is the one rule that **outranks every other rule** in the model:

- it drops the friendship edge in both directions (`block` calls `unfriend`),
- `can_see` checks it **first**, before privacy is even consulted,
- and `add_friend` refuses to re-create the edge while a block is in place.

> **Design note — silent refusal.** `User.add_friend` currently `return`s quietly when the
> pair is blocked (or when you try to friend yourself). That is a deliberate choice for a
> *privacy* rule: telling Bob "you can't friend Alice because she blocked you" leaks the block.
> Compare with `Post.react`, where an invalid call *should* raise, because there is no secret
> to protect. "Fail loud" is the default; "fail silent" needs a reason like this one.

In [ ]:
# Carol blocks Alice. Now neither can see the other's posts,
# and their friendship is dropped.
carol.block(alice)
assert alice not in carol.friends
assert carol not in alice.friends
assert not can_see(alice, p3)        # Alice can't see Carol's post
assert not can_see(carol, p1)        # Carol can't see Alice's post
print('blocking works both ways')
# Undo for next cell
carol.unblock(alice); carol.add_friend(alice)


## Step 8 (bonus) — Fanout-on-write variant

For *normal* users (thousands of friends, not millions), pre-pushing new posts into each friend's
"inbox" makes reads O(1) on the number of posts. The tradeoff: every post costs O(#friends) writes, and you now have to
keep the inbox in sync with privacy/block changes.

This class shows the idea in ~15 lines — same public API as `NewsFeed`.


In [ ]:
class FanoutNewsFeed:
    '''Push-based: when a post is created, copy it into every friend's inbox.'''
    def __init__(self):
        self._inbox: dict[int, list[Post]] = {}   # user.id -> posts

    def publish(self, post: Post) -> None:
        audience = {post.author} | post.author.friends
        for u in audience:
            self._inbox.setdefault(u.id, []).append(post)

    def for_user(self, user: User, limit: int = 10) -> list[Post]:
        # Still re-check visibility at read time (privacy can change after publish).
        inbox = self._inbox.get(user.id, [])
        return sorted(
            (p for p in inbox if can_see(user, p)),
            key=lambda p: p.ts, reverse=True,
        )[:limit]

pushfeed = FanoutNewsFeed()
for p in posts:
    pushfeed.publish(p)

assert {p.id for p in pushfeed.for_user(alice)} == {p.id for p in feed.for_user(alice)}
print('fanout-on-write produces same feed as fanout-on-read for this small graph')


## Step 9 — Verify the design

Prints prove the demo ran. **Assertions prove the design holds.** Each line below is one
sentence of the spec, written so it fails loudly the moment someone breaks it.

In [ ]:
# --- Friendship is symmetric, idempotent, and never self-directed -----------
x, y = User("X"), User("Y")
x.add_friend(y); x.add_friend(y)                 # twice on purpose
assert x.friends == {y} and y.friends == {x},    "friendship must mirror on both sides"
x.add_friend(x)
assert x not in x.friends,                       "nobody is their own friend"
x.unfriend(y)
assert not x.friends and not y.friends,          "unfriend must clear BOTH sides"

# --- Blocking outranks friendship and cannot be bypassed -------------------
x.add_friend(y)
x.block(y)
assert y not in x.friends and x not in y.friends, "block implies unfriend, both ways"
y.add_friend(x)                                   # the blocked side tries again
assert x not in y.friends,                        "a blocked pair cannot re-friend"
x.unblock(y); x.add_friend(y)
assert y in x.friends,                            "unblock restores the ability to friend"

# --- One reaction per user per post (dict upsert, not list append) ---------
post = Post(x, "invariant check", privacy=Privacy.PUBLIC)
for r in (ReactionType.LIKE, ReactionType.LOVE, ReactionType.LOVE):
    post.react(y, r)
assert post.reaction_summary() == {ReactionType.LOVE: 1}, "re-reacting replaces, never adds"
post.remove_reaction(y)
assert post.reaction_summary() == {},              "removing a reaction leaves no ghost"

# --- can_see: the full privacy truth table --------------------------------
stranger = User("Stranger")
cases = [
    #  privacy,             viewer,    expected
    (Privacy.ONLY_ME,  x,         True),    # author always sees their own
    (Privacy.ONLY_ME,  y,         False),   # even a friend does not
    (Privacy.FRIENDS,  y,         True),    # friend sees friends-only
    (Privacy.FRIENDS,  stranger,  False),   # stranger does not
    (Privacy.PUBLIC,   stranger,  True),    # stranger sees public
]
for privacy, viewer, expected in cases:
    p_ = Post(x, "t", privacy=privacy)
    assert can_see(viewer, p_) is expected, f"{privacy} / {viewer} should be {expected}"

# Blocking beats even PUBLIC — the check runs before privacy.
x.block(stranger)
assert not can_see(stranger, Post(x, "t", privacy=Privacy.PUBLIC)), "block beats PUBLIC"
x.unblock(stranger)

# --- Feed: newest-first, own posts included, non-friends excluded ----------
feed_posts = [Post(x, "old", ts=now - timedelta(minutes=9)),
              Post(y, "new", ts=now - timedelta(minutes=1)),
              Post(stranger, "unrelated", ts=now, privacy=Privacy.PUBLIC)]
nf = NewsFeed(feed_posts)
got = nf.for_user(x)
assert [p_.content for p_ in got] == ["new", "old"], "feed is newest-first"
assert all(p_.author in {x} | x.friends for p_ in got), "feed never leaks non-friends"
assert len(nf.for_user(x, limit=1)) == 1,               "limit is honoured"

print("all design invariants hold ✅")

## 🧪 Try it yourself

1. **Pages** — add a `Page` class users can *follow* (one-way), unlike symmetric `Friend`.
   Pages should appear in followers' feeds but cannot send friend requests.
2. **Groups** — a `Group` has members; posts addressed to a group are visible only to members.
3. **Nested comments** — let a `Comment` have its own `comments: list[Comment]`. Careful with depth.
4. **Hybrid fanout** — fanout-on-write for normal users, fanout-on-read for posts by "celebrities"
   (authors with > N friends/followers). Real networks do exactly this.
5. **"People you may know"** — friends-of-friends minus existing friends, ranked by number of mutuals.
